# 02 · Classical pipeline: HOG + Linear SVM (from scratch)

**SIBOL: System for Image-Based Observation and Learning**, Milestone 2

Approach 1 of the comparative study: hand-engineered **Histogram of Oriented Gradients** features and a **one-vs-rest linear SVM**, both written in NumPy in `src/classical/`. No ready-made HOG or SVM is used. scikit-learn is used only for evaluation metrics.

### Protocol (official Flowers-102 splits, never mixed)
| Split | Images | Used for |
|---|---:|---|
| train | 1,020 | fitting the feature standardiser and the SVM |
| val | 1,020 | choosing hyperparameters (by **macro-F1**) |
| test | 6,149 | **one** final evaluation of the selected model |

### How to run in a fresh Google Colab session
1. The three source files must be in `MyDrive/sibol/data/` (`102flowers.tgz`, `imagelabels.mat`, `setid.mat`).
2. The SIBOL code with `src/classical/` must be reachable: pushed to the GitHub repository in `REPO_URL`, **or** copied to Drive (then set `CODE_DIR_COLAB` to that folder).
3. **Runtime → Run all.** A GPU is **not** needed: everything here runs on the CPU. A CPU runtime is fine.

### Where things are stored
| What | Where (Colab) | Lifetime |
|---|---|---|
| Source files | `MyDrive/sibol/data/` | persistent |
| Extracted images | `/content/flowers102` | wiped when the runtime ends |
| HOG feature cache (~265 MB) | `MyDrive/sibol/features/hog/` | persistent, re-used on the next run |
| Model and standardiser | `MyDrive/sibol/checkpoints/` | persistent |
| Metrics (JSON/CSV) | `MyDrive/sibol/outputs/metrics/` | persistent |
| Figures (PNG) | `MyDrive/sibol/outputs/figures/` | persistent |

With `DOWNLOAD = False` (the default) nothing is downloaded: a missing source file stops the notebook with instructions.

## 1. Environment setup

### 1.1 Configuration
**All paths and options are set here.** Later cells use only these values.

In [ ]:
from pathlib import Path

# ---- SIBOL code (Colab) ------------------------------------------------------
REPO_URL = "https://github.com/didyouseejay/sibol.git"
CODE_DIR_COLAB = Path("/content/sibol")  # clone target; or a Drive copy, e.g. Path("/content/drive/MyDrive/sibol_code")

# ---- Google Drive (Colab) ----------------------------------------------------
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "sibol"
SOURCE_DIR_COLAB = DRIVE_PROJECT_DIR / "data"
FEATURES_DIR_COLAB = DRIVE_PROJECT_DIR / "features" / "hog"
CHECKPOINTS_DIR_COLAB = DRIVE_PROJECT_DIR / "checkpoints"
METRICS_DIR_COLAB = DRIVE_PROJECT_DIR / "outputs" / "metrics"
FIGURES_DIR_COLAB = DRIVE_PROJECT_DIR / "outputs" / "figures"

# ---- Colab temporary storage (fast local disk, wiped when the runtime ends) ----
WORK_ROOT_COLAB = Path("/content/flowers102")

# ---- Local runs (relative to the project root; git-ignored except small metrics/figures) ----
SOURCE_SUBDIR_LOCAL = "data"
WORK_SUBDIR_LOCAL = "data/extracted"
FEATURES_SUBDIR_LOCAL = "features/hog"
CHECKPOINTS_SUBDIR_LOCAL = "checkpoints"
METRICS_SUBDIR_LOCAL = "outputs/metrics"
FIGURES_SUBDIR_LOCAL = "outputs/figures"

# ---- Data behaviour ----------------------------------------------------------------
DOWNLOAD = False          # True = allow downloading MISSING source files from the official site
VERIFY_MD5 = True         # check source files against the official MD5 checksums
FORCE_REEXTRACT = False   # True = ignore valid feature caches and recompute HOG

# ---- Preprocessing and HOG (Milestone 2 configuration) -----------------------------
IMAGE_SIZE = 128          # direct resize to 128 x 128 (aspect ratio NOT preserved)
CELL_SIZE = 8
N_BINS = 9                # over 0-180 degrees (unsigned)
BLOCK_SIZE = 2            # cells per block side
BLOCK_STRIDE = 1          # cells
L2HYS_CLIP = 0.2

# ---- Linear SVM: validation grid (18 configurations) ----------------------------------
REG_GRID = [1e-4, 1e-3, 1e-2]          # lambda (L2 on weights, bias not regularised)
LR_GRID = [1e-4, 1e-3, 1e-2]           # initial learning rate
POSITIVE_WEIGHT_GRID = [None, "balanced"]  # None = unweighted; "balanced" = n_neg / n_pos (= 101)
N_EPOCHS = 50
BATCH_SIZE = 128
LR_DECAY = 0.05           # eta_t = eta_0 / (1 + LR_DECAY * epoch)
SEED = 42                 # SVM shuffling order (training is reproducible)

# ---- Demo / timing -------------------------------------------------------------------
DEMO_INDEX = 0            # training image used to illustrate HOG
N_TIMING_IMAGES = 200     # images used to time the HOG computation alone

### 1.2 Runtime and Google Drive
Detects Colab and mounts Google Drive explicitly. Locally, this step is skipped.

In [ ]:
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    if not DRIVE_PROJECT_DIR.is_dir():
        raise FileNotFoundError(
            f"Google Drive is mounted, but the project folder {DRIVE_PROJECT_DIR} does not exist. "
            "Create it (or run notebooks/00_environment_setup.ipynb) and put the dataset files in its 'data' folder."
        )
    print(f"Google Drive mounted at {DRIVE_MOUNT_POINT}")
else:
    print("Running locally: Google Drive mount skipped.")

### 1.3 Load the SIBOL project code
In Colab the repository is cloned into temporary storage if it is not already there. The notebook then checks that the code on disk contains the Milestone 2 modules, so an old copy left from an earlier session is never used by mistake.

In [ ]:
import subprocess
import sys

REQUIRED = [Path("src") / "data" / "flowers102.py", Path("src") / "classical" / "hog.py",
            Path("src") / "classical" / "linear_svm.py", Path("src") / "classical" / "features.py",
            Path("src") / "evaluation" / "metrics.py"]

def missing_modules(root):
    return [str(p) for p in REQUIRED if not (root / p).is_file()]

if IN_COLAB:
    PROJECT_ROOT = CODE_DIR_COLAB
    if not (PROJECT_ROOT / REQUIRED[0]).is_file():
        if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
            raise FileNotFoundError(
                f"{PROJECT_ROOT} exists but does not contain the SIBOL code. Delete it "
                f"(e.g. run: !rm -rf {PROJECT_ROOT}) and re-run this cell, or set CODE_DIR_COLAB."
            )
        print(f"Cloning {REPO_URL} into {PROJECT_ROOT} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / REQUIRED[0]).is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

missing = missing_modules(PROJECT_ROOT)
if missing:
    raise FileNotFoundError(
        f"The SIBOL code in {PROJECT_ROOT} is missing Milestone 2 files: {missing}. It is probably an older "
        f"copy. In Colab: run !rm -rf {PROJECT_ROOT} and re-run this cell (make sure the M2 code has been "
        "pushed to GitHub first), or point CODE_DIR_COLAB to an up-to-date copy on Drive."
    )

# Forget any previously imported 'src' package so we import the files on disk now.
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import flowers102 as f102
from src.classical import features as feats
from src.classical import hog as hogmod
from src.classical.hog import HOGConfig
from src.classical.linear_svm import LinearSVMOvR, SVMConfig
from src.classical.preprocessing import FeatureStandardizer, PreprocessConfig, preprocess_image
from src.evaluation import metrics as M
from src.evaluation import plots as P

commit = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"Project root : {PROJECT_ROOT}")
print(f"Git commit   : {commit or 'n/a (not a git checkout)'}")

### 1.4 Resolve paths and libraries

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np

try:
    from tqdm.auto import tqdm
except ImportError:  # progress bars are optional
    tqdm = None

if IN_COLAB:
    SOURCE_DIR, WORK_ROOT = SOURCE_DIR_COLAB, WORK_ROOT_COLAB
    FEATURES_DIR, CHECKPOINTS_DIR = FEATURES_DIR_COLAB, CHECKPOINTS_DIR_COLAB
    METRICS_DIR, FIGURES_DIR = METRICS_DIR_COLAB, FIGURES_DIR_COLAB
else:
    SOURCE_DIR = PROJECT_ROOT / SOURCE_SUBDIR_LOCAL
    WORK_ROOT = PROJECT_ROOT / WORK_SUBDIR_LOCAL
    FEATURES_DIR = PROJECT_ROOT / FEATURES_SUBDIR_LOCAL
    CHECKPOINTS_DIR = PROJECT_ROOT / CHECKPOINTS_SUBDIR_LOCAL
    METRICS_DIR = PROJECT_ROOT / METRICS_SUBDIR_LOCAL
    FIGURES_DIR = PROJECT_ROOT / FIGURES_SUBDIR_LOCAL

for d in (FEATURES_DIR, CHECKPOINTS_DIR, METRICS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

ENV = {**M.environment_info(), "runtime": "Google Colab" if IN_COLAB else "Local", "git_commit": commit or None}
print(f"Runtime       : {ENV['runtime']}")
print(f"Source files  : {SOURCE_DIR}")
print(f"Extracted data: {WORK_ROOT / f102.TORCHVISION_SUBDIR}")
print(f"Feature cache : {FEATURES_DIR}")
print(f"Checkpoints   : {CHECKPOINTS_DIR}")
print(f"Metrics       : {METRICS_DIR}")
print(f"Figures       : {FIGURES_DIR}")
print(f"DOWNLOAD      : {DOWNLOAD}")
print(f"NumPy {ENV['numpy']}, scikit-learn {ENV['scikit_learn']}, Python {ENV['python']}")

## 2. Dataset loading

Same steps as notebook 01, through the project loader `src/data/flowers102.py`: verify the source files, extract the images to temporary storage, load the **official** splits, and run the 10 integrity checks. The notebook stops if any check fails.

In [ ]:
if DOWNLOAD:
    fetched = f102.download_source_files(SOURCE_DIR)
    print(f"Downloaded: {fetched or 'nothing (all files already present)'}")
else:
    missing = f102.missing_source_files(SOURCE_DIR)
    if missing:
        raise FileNotFoundError(f102.missing_files_message(SOURCE_DIR, missing))

if VERIFY_MD5:
    print("Checking MD5 checksums (reading the 345 MB archive from Drive may take a few minutes)...")
source_report = f102.verify_source_files(SOURCE_DIR, check_md5=VERIFY_MD5)
for name, info in source_report.items():
    md5 = {True: "MD5 OK", False: "MD5 MISMATCH", None: "MD5 not checked"}[info["md5_ok"]]
    print(f"  {name:<16} {info['size']:>12,} bytes   {md5}")

reused = f102.is_prepared(WORK_ROOT)
t0 = time.time()
f102.prepare_dataset(SOURCE_DIR, WORK_ROOT, download=False, check_md5=False)  # already verified above
print("Reused existing extraction." if reused else f"Extracted images in {time.time() - t0:.0f} s.")

In [ ]:
datasets = {split: f102.load_split(WORK_ROOT, split) for split in f102.SPLITS}
labels = {s: f102.get_labels(ds) for s, ds in datasets.items()}
files = {s: f102.get_image_files(ds) for s, ds in datasets.items()}
image_ids = {s: feats.image_ids_from_files(files[s]) for s in f102.SPLITS}

checks = f102.integrity_checks(labels, files)
for i, (name, passed, detail) in enumerate(checks, start=1):
    print(f"{i:>2}. [{'PASS' if passed else 'FAIL'}] {name}: {detail}")
n_passed = sum(passed for _, passed, _ in checks)
print(f"\n{n_passed}/10 checks passed.")
if n_passed != 10:
    raise RuntimeError("Integrity checks failed: investigate before using this data.")

CLASS_NAMES = getattr(datasets["train"], "classes", None) or [f"class_{i:03d}" for i in range(f102.NUM_CLASSES)]
for s in f102.SPLITS:
    print(f"{s:<5}: {len(image_ids[s]):>5} images, first {image_ids[s][0]}, last {image_ids[s][-1]}")

## 3. HOG configuration

| Setting | Value |
|---|---|
| Input | RGB, resized **directly** to 128 × 128 (aspect ratio not preserved), then grayscale (BT.601) in [0, 1] |
| Gradient | centred [-1, 0, 1] differences, replicated borders |
| Orientation | unsigned, 0–180°, 9 bins of 20° (centres 10°, 30°, …, 170°), linear interpolation between neighbouring bins |
| Cells | 8 × 8 px → 16 × 16 cells |
| Blocks | 2 × 2 cells, stride 1 cell → 15 × 15 blocks |
| Normalisation | L2-Hys (L2, clip at 0.2, L2) |
| Length | 15 × 15 × 2 × 2 × 9 = **8,100** |

**Limitation:** image aspect ratios range from about 0.58 to 1.71 (notebook 01), so the direct resize stretches or squashes many flowers. This is deliberate for this milestone.

In [ ]:
PRE_CFG = PreprocessConfig(image_size=IMAGE_SIZE)
HOG_CFG = HOGConfig(cell_size=CELL_SIZE, n_bins=N_BINS, block_size=BLOCK_SIZE, block_stride=BLOCK_STRIDE,
                    clip=L2HYS_CLIP)
(cells_yx, blocks_yx) = HOG_CFG.grid_shape((IMAGE_SIZE, IMAGE_SIZE))
FEATURE_LENGTH = HOG_CFG.feature_length((IMAGE_SIZE, IMAGE_SIZE))

print("Preprocessing:", PRE_CFG.to_dict())
print("HOG          :", HOG_CFG.to_dict())
print(f"Cells {cells_yx}, blocks {blocks_yx}, feature length {FEATURE_LENGTH:,}")
assert FEATURE_LENGTH == 8100, "Milestone 2 configuration must give 8,100 features"

## 4. HOG implementation demonstration

One training image goes through every step of `src/classical/hog.py`. The last panel draws each cell's histogram as line segments along the **edge** direction (perpendicular to the gradient); brighter = stronger.

The cell below also times the HOG computation alone (on already-preprocessed images), separately from JPEG decoding and resizing.

In [ ]:
demo_image, demo_label = datasets["train"][DEMO_INDEX]
gray = preprocess_image(demo_image, PRE_CFG)
gx, gy = hogmod.compute_gradients(gray)
magnitude, orientation = hogmod.magnitude_orientation(gx, gy)
cell_hist = hogmod.cell_histograms(magnitude, orientation, HOG_CFG)
blocks = hogmod.build_blocks(cell_hist, HOG_CFG)
normed = hogmod.l2_hys(blocks.reshape(blocks.shape[0], blocks.shape[1], -1), HOG_CFG.clip, HOG_CFG.eps)
vector = hogmod.hog_features(gray, HOG_CFG)

print(f"image {image_ids['train'][DEMO_INDEX]} ({CLASS_NAMES[demo_label]}), original size {demo_image.size}")
print(f"1. grayscale        {gray.shape}, range [{gray.min():.3f}, {gray.max():.3f}]")
print(f"2. gradients        gx {gx.shape}, gy {gy.shape}")
print(f"3. magnitude/angle  max magnitude {magnitude.max():.3f}, angles in [{orientation.min():.1f}, {orientation.max():.1f}]")
print(f"4. cell histograms  {cell_hist.shape}")
print(f"5. blocks           {blocks.shape}")
print(f"6. L2-Hys blocks    {normed.shape}, max value {normed.max():.3f}")
print(f"7. feature vector   {vector.shape}, finite: {np.isfinite(vector).all()}")
assert np.allclose(normed.ravel(), vector)

P.plot_hog_demo(np.asarray(demo_image), gray, magnitude, cell_hist, HOG_CFG.cell_size,
                title=f"HOG steps: {CLASS_NAMES[demo_label]}", path=FIGURES_DIR / "hog_demo.png")
plt.show()

timing_grays = [preprocess_image(datasets["train"][i][0], PRE_CFG) for i in range(N_TIMING_IMAGES)]
t0 = time.perf_counter()
for g in timing_grays:
    hogmod.hog_features(g, HOG_CFG)
HOG_ONLY_SECONDS_PER_IMAGE = (time.perf_counter() - t0) / N_TIMING_IMAGES
print(f"HOG computation alone: {HOG_ONLY_SECONDS_PER_IMAGE * 1000:.2f} ms per image ({N_TIMING_IMAGES} images)")

## 5. Feature extraction

Every image of every split goes through preprocessing and HOG. Images are read with the official loader (`datasets[split][i]`), and each label returned by the loader is checked against the split's label list, so features, image IDs and labels stay aligned.

The extraction time recorded here includes **JPEG decoding + resizing + grayscale + HOG**. On the first run this takes several minutes; later runs load the cache (Section 6) in seconds.

In [ ]:
feature_sets = {}
for split in f102.SPLITS:
    ds = datasets[split]
    fs, status = feats.get_or_extract_features(
        split, image_ids[split], labels[split], ds.__getitem__, FEATURES_DIR, PRE_CFG, HOG_CFG,
        force=FORCE_REEXTRACT, progress=(lambda r, s=split: tqdm(r, desc=f"HOG {s}")) if tqdm else None,
    )
    feature_sets[split] = fs
    print(f"{split:<5}: {fs.features.shape}  {status}")
    print(f"       extraction took {fs.meta['extraction_seconds']:.1f} s "
          f"({fs.meta['extraction_seconds_per_image'] * 1000:.1f} ms per image, decoding included)")

## 6. Feature caching

One file per split, `hog_<split>_<config-hash>.npz`, in `FEATURES_DIR` (on Drive in Colab, git-ignored locally). A cache is used only if its metadata matches the current preprocessing and HOG configuration, the split name, the number of samples and a SHA-256 fingerprint of the ordered (image ID, label) list. Otherwise it is rebuilt automatically.

This section verifies the caches: it reloads them, checks that the splits are disjoint and have the official sizes, and checks that every feature is finite.

In [ ]:
for path in sorted(FEATURES_DIR.glob("hog_*.npz")):
    print(f"  {path.name:<32} {path.stat().st_size / 1e6:8.1f} MB")

# Reload every split from disk and compare with what is in memory.
for split in f102.SPLITS:
    expected = feats.expected_meta(split, image_ids[split], labels[split], PRE_CFG, HOG_CFG)
    reloaded, reason = feats.load_feature_cache(feats.cache_path(FEATURES_DIR, split, PRE_CFG, HOG_CFG), expected)
    assert reloaded is not None, f"{split}: cache could not be reloaded ({reason})"
    assert np.array_equal(reloaded.features, feature_sets[split].features)
    assert reloaded.image_ids.tolist() == image_ids[split] and reloaded.labels.tolist() == labels[split]
    print(f"{split:<5}: cache valid ({reason}); config hash {feats.config_hash(PRE_CFG, HOG_CFG)}")

feats.assert_disjoint_splits(feature_sets, expected_sizes=f102.EXPECTED_SPLIT_SIZES)
assert all(np.isfinite(fs.features).all() for fs in feature_sets.values())
print("Splits are disjoint, have the official sizes, and all features are finite.")

## 7. Linear SVM implementation

`src/classical/linear_svm.py` trains 102 one-vs-rest binary SVMs together, as a weight matrix **W** (8,100 × 102) and a bias vector **b** (102). For class *c*, with *y* = +1 for the class and −1 for the rest:

$$J_c(\mathbf w, b) = \frac{\lambda}{2}\lVert\mathbf w\rVert^2 + \frac1N\sum_{i=1}^{N} s_{ic}\,\max\bigl(0,\;1 - y_{ic}(\mathbf w^\top\mathbf x_i + b)\bigr)$$

* hinge loss; L2 regularisation on **w** only (**the bias is not regularised**);
* sample weight $s_{ic}$ = `positive_weight` for the class's own images and 1 for all others. `"balanced"` sets it to $n_\text{neg}/n_\text{pos} = 1010/10 = 101$, so positives and negatives carry equal total weight;
* mini-batch subgradient descent, seeded shuffling, step size $\eta_t = \eta_0/(1 + \text{decay}\cdot t)$;
* prediction = class with the highest score $\mathbf x^\top W + \mathbf b$.

Parameters: 8,100 × 102 + 102 = **826,302**.

**Feature standardisation** (zero mean, unit variance per feature) is fitted on the **training features only** and then applied unchanged to validation and test.

In [ ]:
scaler = FeatureStandardizer().fit(feature_sets["train"].features)
X = {"train": scaler.transform(feature_sets["train"].features, dtype=np.float64)}
X["val"] = scaler.transform(feature_sets["val"].features, dtype=np.float32)  # float32 saves memory
# The test features are standardised only in Section 10 (the single test evaluation).
y = {s: feature_sets[s].labels for s in f102.SPLITS}

assert scaler.n_fit_samples_ == f102.EXPECTED_SPLIT_SIZES["train"]
print(f"Standardiser fitted on {scaler.n_fit_samples_} training images only.")
for s in ("train", "val"):
    print(f"  {s:<5} mean of features after scaling: {X[s].mean():+.4f}, std {X[s].std():.4f}")
print(f"Expected parameter count: {FEATURE_LENGTH} x {f102.NUM_CLASSES} + {f102.NUM_CLASSES} = "
      f"{FEATURE_LENGTH * f102.NUM_CLASSES + f102.NUM_CLASSES:,}")

## 8. Model training (hyperparameter search)

Every configuration in the grid (`REG_GRID` × `LR_GRID` × `POSITIVE_WEIGHT_GRID`, 18 in total) is trained on the **training split** with the same seed and scored on the **validation split**. The test split is not touched.

Selection rule: highest **validation macro-F1**; ties go to higher validation top-1 accuracy, then to the earlier grid entry. Only the selected model is kept in memory.

In [ ]:
grid_rows, histories = [], {}
best = None
t_grid = time.perf_counter()
for pw in POSITIVE_WEIGHT_GRID:
    for lr in LR_GRID:
        for reg in REG_GRID:
            cfg = SVMConfig(learning_rate=lr, reg_strength=reg, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE,
                            lr_decay=LR_DECAY, positive_weight=pw, seed=SEED)
            model = LinearSVMOvR(cfg).fit(X["train"], y["train"])
            val_m = M.classification_metrics(y["val"], model.predict(X["val"]), f102.NUM_CLASSES)
            train_acc = float((model.predict(X["train"]) == y["train"]).mean())
            row = {"positive_weight": str(pw), "learning_rate": lr, "reg_strength": reg,
                   "fit_seconds": model.history.fit_seconds, "final_objective": model.history.objective[-1],
                   "train_top1_accuracy": train_acc,
                   **{f"val_{k}": v for k, v in val_m.items() if k != "n_samples"}}
            grid_rows.append(row)
            histories[(str(pw), lr, reg)] = model.history.objective
            key = (row["val_macro_f1"], row["val_top1_accuracy"])
            if best is None or key > best[0]:
                best = (key, row, model)
            print(f"pw={str(pw):<9} lr={lr:<7g} reg={reg:<7g} | fit {row['fit_seconds']:5.1f} s | "
                  f"train acc {train_acc:.3f} | val acc {row['val_top1_accuracy']:.3f} | "
                  f"val macro-F1 {row['val_macro_f1']:.3f}")
GRID_SECONDS = time.perf_counter() - t_grid
_, best_row, best_model = best
print(f"\nGrid of {len(grid_rows)} configurations took {GRID_SECONDS / 60:.1f} min.")
M.save_rows_csv(grid_rows, METRICS_DIR / "hog_svm_validation_grid.csv")

## 9. Validation

The selected configuration and its validation metrics. The best **unweighted** and best **balanced** configurations are compared directly; weighting is kept only if it actually scores higher on validation.

In [ ]:
print("Selected configuration:")
for k in ("positive_weight", "learning_rate", "reg_strength"):
    print(f"  {k:<16} {best_row[k]}")
VAL_METRICS = M.classification_metrics(y["val"], best_model.predict(X["val"]), f102.NUM_CLASSES)
print("Validation metrics of the selected model:")
for k, v in VAL_METRICS.items():
    print(f"  {k:<16} {v:.4f}" if isinstance(v, float) else f"  {k:<16} {v}")

WEIGHTING_COMPARISON = {}
for pw in sorted({r["positive_weight"] for r in grid_rows}):
    rows_pw = [r for r in grid_rows if r["positive_weight"] == pw]
    top = max(rows_pw, key=lambda r: (r["val_macro_f1"], r["val_top1_accuracy"]))
    WEIGHTING_COMPARISON[pw] = {k: top[k] for k in ("learning_rate", "reg_strength", "val_macro_f1",
                                                     "val_top1_accuracy")}
    print(f"Best with positive_weight={pw:<9}: val macro-F1 {top['val_macro_f1']:.4f}, "
          f"val acc {top['val_top1_accuracy']:.4f} (lr {top['learning_rate']:g}, reg {top['reg_strength']:g})")

assert best_model.n_parameters == FEATURE_LENGTH * f102.NUM_CLASSES + f102.NUM_CLASSES
print(f"Learned parameters: {best_model.n_parameters:,} "
      f"(W {best_model.W.shape[0]} x {best_model.W.shape[1]} + b {best_model.b.shape[0]})")

best_model.save(CHECKPOINTS_DIR / "hog_svm_best.npz")
scaler.save(CHECKPOINTS_DIR / "hog_svm_standardizer.npz")
print(f"Saved model and standardiser to {CHECKPOINTS_DIR}")

## 10. Test evaluation

The selected model is evaluated on the official test split **once**. Nothing from this section feeds back into any choice above.

Inference time = standardise + score all 102 classes + argmax, measured over the 6,149 cached test features. HOG extraction time is reported separately (Section 5).

In [ ]:
if globals().get("TEST_EVALUATED"):
    raise RuntimeError("The test split was already evaluated in this session. Re-running would invite tuning "
                       "on test; restart the runtime to repeat the whole experiment from scratch.")

t0 = time.perf_counter()
X_test_scaled = scaler.transform(feature_sets["test"].features, dtype=np.float32)
test_scores = best_model.decision_function(X_test_scaled)
test_pred = best_model.classes_[np.argmax(test_scores, axis=1)]
INFER_SECONDS = time.perf_counter() - t0
TEST_EVALUATED = True

assert test_scores.shape == (f102.EXPECTED_SPLIT_SIZES["test"], f102.NUM_CLASSES)
TEST_METRICS = M.classification_metrics(y["test"], test_pred, f102.NUM_CLASSES)
PER_CLASS = M.per_class_metrics(y["test"], test_pred, CLASS_NAMES)
CM = M.confusion(y["test"], test_pred, f102.NUM_CLASSES)

print("Test metrics:")
for k, v in TEST_METRICS.items():
    print(f"  {k:<16} {v:.4f}" if isinstance(v, float) else f"  {k:<16} {v}")
print(f"SVM inference: {INFER_SECONDS / len(test_pred) * 1e6:.1f} microseconds per image")

timing = {
    "hog_extraction_seconds": {s: feature_sets[s].meta["extraction_seconds"] for s in f102.SPLITS},
    "hog_extraction_seconds_per_image": {s: feature_sets[s].meta["extraction_seconds_per_image"] for s in f102.SPLITS},
    "hog_extraction_note": "decode + resize + grayscale + HOG, measured when the cache was created",
    "hog_only_seconds_per_image": HOG_ONLY_SECONDS_PER_IMAGE,
    "svm_training_seconds_selected_config": best_model.history.fit_seconds,
    "svm_grid_search_seconds_total": GRID_SECONDS,
    "svm_inference_seconds_per_image": INFER_SECONDS / len(test_pred),
    "svm_inference_note": "standardise + 102 scores + argmax on cached features",
    "end_to_end_seconds_per_image_estimate":
        feature_sets["test"].meta["extraction_seconds_per_image"] + INFER_SECONDS / len(test_pred),
}
RESULTS = {
    "model": "HOG (from scratch) + one-vs-rest linear SVM (from scratch)",
    "protocol": "train on official train split, select on val by macro-F1, evaluate once on test",
    "split_sizes": {s: len(feature_sets[s]) for s in f102.SPLITS},
    "preprocessing": PRE_CFG.to_dict(),
    "hog": HOG_CFG.to_dict(),
    "feature_length": FEATURE_LENGTH,
    "feature_cache_config_hash": feats.config_hash(PRE_CFG, HOG_CFG),
    "grid": {"reg_strength": REG_GRID, "learning_rate": LR_GRID,
             "positive_weight": [str(p) for p in POSITIVE_WEIGHT_GRID],
             "n_epochs": N_EPOCHS, "batch_size": BATCH_SIZE, "lr_decay": LR_DECAY, "seed": SEED},
    "selected_config": best_model.config.to_dict(),
    "weighting_comparison_val": WEIGHTING_COMPARISON,
    "n_parameters": best_model.n_parameters,
    "validation": VAL_METRICS,
    "test": TEST_METRICS,
    "timing": timing,
    "environment": ENV,
    "created": time.strftime("%Y-%m-%d %H:%M:%S %Z"),
}
M.save_json(RESULTS, METRICS_DIR / "hog_svm_results.json")
M.save_rows_csv(PER_CLASS, METRICS_DIR / "hog_svm_test_per_class.csv")
M.save_matrix_csv(CM, METRICS_DIR / "hog_svm_test_confusion_matrix.csv", CLASS_NAMES)
print(f"Results saved to {METRICS_DIR}")

## 11. Visualizations

Saved as PNG to `FIGURES_DIR`: training curve of the selected model, the validation grid, the row-normalised test confusion matrix, and the classes with the lowest and highest test F1.

In [ ]:
sel = (best_row["positive_weight"], best_row["learning_rate"], best_row["reg_strength"])
P.plot_training_curve(histories[sel], path=FIGURES_DIR / "hog_svm_training_curve.png",
                      title=f"Training objective (pw={sel[0]}, lr={sel[1]:g}, λ={sel[2]:g})")
plt.show()
P.plot_validation_grid(grid_rows, path=FIGURES_DIR / "hog_svm_validation_grid.png",
                       title="HOG + linear SVM: validation macro-F1 per configuration")
plt.show()
P.plot_confusion_matrix(CM, path=FIGURES_DIR / "hog_svm_confusion_matrix.png",
                        title="HOG + linear SVM: test confusion matrix (row-normalised)")
plt.show()
P.plot_per_class_f1(PER_CLASS, path=FIGURES_DIR / "hog_svm_per_class_f1.png",
                    title="HOG + linear SVM: per-class test F1")
plt.show()

## 12. Results and observations

In [ ]:
print(f"Code commit             : {ENV['git_commit']}")
print(f"Selected configuration  : pw={best_row['positive_weight']}, lr={best_row['learning_rate']:g}, "
      f"reg={best_row['reg_strength']:g}, {N_EPOCHS} epochs, batch {BATCH_SIZE}")
print(f"Validation macro-F1     : {VAL_METRICS['macro_f1']:.4f}   top-1 {VAL_METRICS['top1_accuracy']:.4f}")
print(f"Test top-1 accuracy     : {TEST_METRICS['top1_accuracy']:.4f}")
print(f"Test macro precision    : {TEST_METRICS['macro_precision']:.4f}")
print(f"Test macro recall       : {TEST_METRICS['macro_recall']:.4f}")
print(f"Test macro-F1           : {TEST_METRICS['macro_f1']:.4f}")
print(f"Learned parameters      : {best_model.n_parameters:,}")
print(f"HOG extraction (test)   : {timing['hog_extraction_seconds_per_image']['test'] * 1000:.1f} ms/image "
      f"(HOG alone {HOG_ONLY_SECONDS_PER_IMAGE * 1000:.2f} ms)")
print(f"SVM training (selected) : {timing['svm_training_seconds_selected_config']:.1f} s; "
      f"grid total {GRID_SECONDS / 60:.1f} min")
print(f"SVM inference           : {timing['svm_inference_seconds_per_image'] * 1e6:.1f} µs/image")
lowest = sorted(PER_CLASS, key=lambda r: r["f1"])[:5]
highest = sorted(PER_CLASS, key=lambda r: -r["f1"])[:5]
print("Lowest-F1 classes       :", [(r["class_name"], round(r["f1"], 3)) for r in lowest])
print("Highest-F1 classes      :", [(r["class_name"], round(r["f1"], 3)) for r in highest])

### Observations

*To be written after the first complete Colab run, from the printed outputs above. Do not fill in numbers that were not produced by a run.* Suggested points:

- Selected configuration, and whether `"balanced"` positive weighting helped or hurt on validation (Section 9).
- Validation vs test metrics (gap, and accuracy vs macro-F1 given the imbalanced test split).
- Training vs validation accuracy (over-fitting with 1,020 training images and 8,100 features).
- Which classes HOG separates well or poorly (Section 11), and plausible reasons (shape vs colour: HOG ignores colour).
- Cost: HOG extraction time, SVM training time, inference time, 826,302 parameters.
- Known limitation: the direct 128 × 128 resize distorts aspect ratios (0.58–1.71).